# Imports

In [13]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from torch.nn import Module
import os
from tqdm import tqdm
import torch
import torchvision
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"
MODELS_PATH = "saved_models"
os.makedirs(MODELS_PATH, exist_ok=True)

# Data

In [14]:
transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761)
        ),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761)
        ),
    ]
)

In [ ]:
# 1. Load the full CIFAR-100 training dataset
full_trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=False, transform=transform_train
)

# 2. Define split sizes (4 equal parts of 12,500 images each)
total_size = len(full_trainset)
split_size = total_size // 4
# Adjust the last split in case total_size isn't perfectly divisible by 4
lengths = [split_size, split_size, split_size, total_size - (3 * split_size)]

# 3. Perform the split
train_subsets = random_split(
    full_trainset, lengths, generator=torch.Generator().manual_seed(42)
)

# 4. Create DataLoaders for each subset
trainloaders = [
    DataLoader(subset, batch_size=128, shuffle=True, num_workers=4)
    for subset in train_subsets
]

# Access them individually:
trainloader_1 = trainloaders[0]
trainloader_2 = trainloaders[1]
trainloader_3 = trainloaders[2]
trainloader_4 = trainloaders[3]

# 5. Test set (standard setup)
testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform_test
)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=4)

/home/akm/miniconda3/envs/convey/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


# Training

In [ ]:
def get_resnet_model():
    model = models.resnet18(weights=None)

    # Replace first conv layer
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # Remove maxpool
    model.maxpool = nn.Identity()

    # Adjust classifier for CIFAR-100
    model.fc = nn.Linear(model.fc.in_features, 100)
    model.to(device)
    return model


model_1 = get_resnet_model()
model_2 = get_resnet_model()
model_3 = get_resnet_model()
model_4 = get_resnet_model()

In [17]:
def train_model(device, trainloader, model_1):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model_1.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[100, 150], gamma=0.1
    )
    epochs = 150
    for epoch in range(epochs):
        model_1.train()

        # Wrap your trainloader with tqdm for a per-epoch progress bar
        loop = tqdm(enumerate(trainloader), total=len(trainloader), leave=False)

        for batch_idx, (images, labels) in loop:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model_1(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Update the progress bar with the current loss
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())

        scheduler.step()


# train_model(device, trainloader_1, model_1)
# torch.save(model_1, os.path.join(MODELS_PATH, "model_1.pth"))

# train_model(device, trainloader_2, model_2)
# torch.save(model_2, os.path.join(MODELS_PATH, "model_2.pth"))

# train_model(device, trainloader_3, model_3)
# torch.save(model_3, os.path.join(MODELS_PATH, "model_3.pth"))

# train_model(device, trainloader_4, model_4)
# torch.save(model_4, os.path.join(MODELS_PATH, "model_4.pth"))

# Test

In [18]:
def test_model(device, test_loader, model) -> float:
    model.eval()
    correct = 0
    total = 0

    # Disable gradient calculation for efficiency and to prevent unintended training
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)

            # Forward pass
            outputs = model(data)

            # Get predictions from the maximum value
            _, predicted = torch.max(outputs.data, 1)

            # Statistics
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = (correct / total) * 100
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [19]:
model_1: Module = torch.load(
    os.path.join(MODELS_PATH, "model_1.pth"), weights_only=False
)

model_2: Module = torch.load(
    os.path.join(MODELS_PATH, "model_2.pth"), weights_only=False
)

model_3: Module = torch.load(
    os.path.join(MODELS_PATH, "model_3.pth"), weights_only=False
)

model_4: Module = torch.load(
    os.path.join(MODELS_PATH, "model_4.pth"), weights_only=False
)

In [20]:
model_1_acc = test_model(device, testloader, model_1)
model_2_acc = test_model(device, testloader, model_2)
model_3_acc = test_model(device, testloader, model_3)
model_4_acc = test_model(device, testloader, model_4)

print(
    f"Model 1 accuracy: {model_1_acc}%, Model 2 accuracy: {model_2_acc}%, Model 3 accuracy: {model_3_acc}%, Model 4 accuracy: {model_4_acc}%"
)

Test Accuracy: 59.44%
Test Accuracy: 59.45%
Test Accuracy: 59.49%
Test Accuracy: 59.81%
Model 1 accuracy: 59.440000000000005%, Model 2 accuracy: 59.45%, Model 3 accuracy: 59.489999999999995%, Model 4 accuracy: 59.809999999999995%


# Merge

In [21]:
from ml.pytorch.merge import TorchGreedySoup

In [22]:
model_1_weights = model_1.state_dict()
model_2_weights = model_2.state_dict()
model_3_weights = model_3.state_dict()
model_4_weights = model_4.state_dict()

In [23]:
import copy

greedy_soup = TorchGreedySoup("trial_CIFAR", weights=model_1_weights)


def merge_test(new_weights):

    new_merged_weights = greedy_soup.merge(new_weights)
    merged_model = copy.deepcopy(model_1)
    merged_model.load_state_dict(new_merged_weights)
    merged_acc = test_model(device, testloader, merged_model)
    print(
        f"Model 1 accuracy: {model_1_acc}%\nModel 2 accuracy: {model_2_acc}%\nModel 3 accuracy: {model_3_acc}%\nModel 4 accuracy: {model_4_acc}%\nMerged accuracy: {merged_acc}%"
    )
    return new_merged_weights

In [24]:
one_two_new_weights = merge_test(model_2_weights)

Test Accuracy: 1.00%
Model 1 accuracy: 59.440000000000005%
Model 2 accuracy: 59.45%
Model 3 accuracy: 59.489999999999995%
Model 4 accuracy: 59.809999999999995%
Merged accuracy: 1.0%


In [25]:
all_merged_weights = merge_test(model_3_weights)

Test Accuracy: 1.00%
Model 1 accuracy: 59.440000000000005%
Model 2 accuracy: 59.45%
Model 3 accuracy: 59.489999999999995%
Model 4 accuracy: 59.809999999999995%
Merged accuracy: 1.0%


In [26]:
all_merged_weights = merge_test(model_4_weights)

Test Accuracy: 1.00%
Model 1 accuracy: 59.440000000000005%
Model 2 accuracy: 59.45%
Model 3 accuracy: 59.489999999999995%
Model 4 accuracy: 59.809999999999995%
Merged accuracy: 1.0%
